In [1]:
import pandas as pd
import os

In [2]:
from langchain_community.document_loaders import CSVLoader


/Users/vishalhanchate/Desktop/Projects/RAG/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
# method 1 Each row becomes a documnets

import csv
from langchain_community.document_loaders import csv_loader


print("1 raw based loader")


csv_loader = CSVLoader(
    file_path="data/structured_files_CSV/products.csv",
    encoding='utf-8',
    csv_args={
        'delimiter': ',',
        'quotechar': '"'
    }
)

csv_docs = csv_loader.load()
print(f"content is : {csv_docs}")
print(f"loaded {len(csv_docs)} documnets")

print("\nFirst Documnet")

print(f"content is : {csv_docs[0].page_content}")
print(f"content is : {csv_docs[0].metadata}")



for idx, doc in enumerate(csv_docs):
    print(f"\nDocument {idx+1}")
    print(doc.page_content)
    print(doc.metadata)



1 raw based loader
content is : [Document(metadata={'source': 'data/structured_files_CSV/products.csv', 'row': 0}, page_content='Product: Laptop\nCategory: Electronics\nPrice: 999.99\nStock: 50\nDescription: High-performance laptop with 16GB RAM and 512GB SSD'), Document(metadata={'source': 'data/structured_files_CSV/products.csv', 'row': 1}, page_content='Product: Mouse\nCategory: Accessories\nPrice: 29.99\nStock: 200\nDescription: Wireless optical mouse with ergonomic design'), Document(metadata={'source': 'data/structured_files_CSV/products.csv', 'row': 2}, page_content='Product: Keyboard\nCategory: Accessories\nPrice: 79.99\nStock: 150\nDescription: Mechanical keyboard with RGB backlighting'), Document(metadata={'source': 'data/structured_files_CSV/products.csv', 'row': 3}, page_content='Product: Monitor\nCategory: Electronics\nPrice: 299.99\nStock: 75\nDescription: 27-inch 4K monitor with HDR support'), Document(metadata={'source': 'data/structured_files_CSV/products.csv', 'row': 

In [13]:
from typing import List
from langchain_core.documents import Document

# Method 2: Custom CSV processing for better control
print("\n2️⃣ Custom CSV Processing")
def process_csv_intelligently(filepath: str) -> List[Document]:
    """Process CSV with intelligent document creation"""
    df = pd.read_csv(filepath)
    documents = []
    
    # Strategy 1: One document per row with structured content
    for idx, row in df.iterrows():
        # Create structured content
        content = f"""Product Information:
        Name: {row['Product']}
        Category: {row['Category']}
        Price: ${row['Price']}
        Stock: {row['Stock']} units
        Description: {row['Description']}"""
        
        # Create document with rich metadata
        doc = Document(
            page_content=content,
            metadata={
                'source': filepath,
                'row_index': idx,
                'product_name': row['Product'],
                'category': row['Category'],
                'price': row['Price'],
                'data_type': 'product_info'
            }
        )
        documents.append(doc)
    return documents


2️⃣ Custom CSV Processing


In [14]:
process_csv_intelligently("/Users/vishalhanchate/Desktop/Projects/RAG/0 - Data_ingest_parsing/data/structured_files_CSV/products.csv")

[Document(metadata={'source': '/Users/vishalhanchate/Desktop/Projects/RAG/0 - Data_ingest_parsing/data/structured_files_CSV/products.csv', 'row_index': 0, 'product_name': 'Laptop', 'category': 'Electronics', 'price': 999.99, 'data_type': 'product_info'}, page_content='Product Information:\n        Name: Laptop\n        Category: Electronics\n        Price: $999.99\n        Stock: 50 units\n        Description: High-performance laptop with 16GB RAM and 512GB SSD'),
 Document(metadata={'source': '/Users/vishalhanchate/Desktop/Projects/RAG/0 - Data_ingest_parsing/data/structured_files_CSV/products.csv', 'row_index': 1, 'product_name': 'Mouse', 'category': 'Accessories', 'price': 29.99, 'data_type': 'product_info'}, page_content='Product Information:\n        Name: Mouse\n        Category: Accessories\n        Price: $29.99\n        Stock: 200 units\n        Description: Wireless optical mouse with ergonomic design'),
 Document(metadata={'source': '/Users/vishalhanchate/Desktop/Projects/RA

## excel processing

In [15]:
# Method 1: Using pandas for full control
print("1️⃣ Pandas-based Excel Processing")
def process_excel_with_pandas(filepath: str) -> List[Document]:
    """Process Excel with sheet awareness"""
    documents = []
    
    # Read all sheets
    excel_file = pd.ExcelFile(filepath)
    
    for sheet_name in excel_file.sheet_names:
        df = pd.read_excel(filepath, sheet_name=sheet_name)
        
        # Create document for each sheet
        sheet_content = f"Sheet: {sheet_name}\n"
        sheet_content += f"Columns: {', '.join(df.columns)}\n"
        sheet_content += f"Rows: {len(df)}\n\n"
        sheet_content += df.to_string(index=False)
        
        doc = Document(
            page_content=sheet_content,
            metadata={
                'source': filepath,
                'sheet_name': sheet_name,
                'num_rows': len(df),
                'num_columns': len(df.columns),
                'data_type': 'excel_sheet'
            }
        )
        documents.append(doc)
    
    return documents

1️⃣ Pandas-based Excel Processing


In [16]:
process_excel_with_pandas("/Users/vishalhanchate/Desktop/Projects/RAG/0 - Data_ingest_parsing/data/structured_files_CSV/inventory.xlsx")

[Document(metadata={'source': '/Users/vishalhanchate/Desktop/Projects/RAG/0 - Data_ingest_parsing/data/structured_files_CSV/inventory.xlsx', 'sheet_name': 'Products', 'num_rows': 5, 'num_columns': 5, 'data_type': 'excel_sheet'}, page_content='Sheet: Products\nColumns: Product, Category, Price, Stock, Description\nRows: 5\n\n Product    Category  Price  Stock                                         Description\n  Laptop Electronics 999.99     50 High-performance laptop with 16GB RAM and 512GB SSD\n   Mouse Accessories  29.99    200        Wireless optical mouse with ergonomic design\nKeyboard Accessories  79.99    150           Mechanical keyboard with RGB backlighting\n Monitor Electronics 299.99     75                 27-inch 4K monitor with HDR support\n  Webcam Electronics  89.99    100                1080p webcam with noise cancellation'),
 Document(metadata={'source': '/Users/vishalhanchate/Desktop/Projects/RAG/0 - Data_ingest_parsing/data/structured_files_CSV/inventory.xlsx', 'sh

In [1]:
from langchain_community.document_loaders import UnstructuredExcelLoader
# Method 2: UnstructuredExcelLoader
print("\n2️⃣ UnstructuredExcelLoader")
try:
    excel_loader = UnstructuredExcelLoader(
        'data/structured_files/inventory.xlsx',
        mode="elements"
    )
    unstructured_docs = excel_loader.load()
    print("  ✅ Handles complex Excel features")
    print("  ✅ Preserves formatting info")
    print("  ❌ Requires unstructured library")
except Exception as e:
    print("  ℹ️ Requires unstructured library with Excel support")

/Users/vishalhanchate/Desktop/Projects/RAG/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



2️⃣ UnstructuredExcelLoader
  ℹ️ Requires unstructured library with Excel support


In [18]:
unstructured_docs

NameError: name 'unstructured_docs' is not defined